In [ ]:
!pip uninstall torch -y
!pip uninstall torchvision -y
!pip install torch==1.6.0
!pip install torchvision==0.7.0
!nvidia-smi

In [ ]:
# Enforce pytorch version 1.6.0
import torch
if torch.__version__ != '1.6.0':
  !pip uninstall torch -y
  !pip uninstall torchvision -y
  !pip install torch==1.6.0
  !pip install torchvision==0.7.0

# Check pytorch version and make sure you use a GPU Kernel
!python -c "import torch; print(torch.__version__)"
!python -c "import torch; print(torch.version.cuda)"
!python --version
!nvidia-smi

In [ ]:
#@title
# Install rdkit
import sys
import os
import requests
import subprocess
import shutil
from logging import getLogger, StreamHandler, INFO


logger = getLogger(__name__)
logger.addHandler(StreamHandler())
logger.setLevel(INFO)


def install(
        chunk_size=4096,
        file_name="Miniconda3-latest-Linux-x86_64.sh",
        url_base="https://repo.continuum.io/miniconda/",
        conda_path=os.path.expanduser(os.path.join("~", "miniconda")),
        rdkit_version=None,
        add_python_path=True,
        force=False):
    """install rdkit from miniconda
    ```
    import rdkit_installer
    rdkit_installer.install()
    ```
    """

    python_path = os.path.join(
        conda_path,
        "lib",
        "python{0}.{1}".format(*sys.version_info),
        "site-packages",
    )

    if add_python_path and python_path not in sys.path:
        logger.info("add {} to PYTHONPATH".format(python_path))
        sys.path.append(python_path)

    if os.path.isdir(os.path.join(python_path, "rdkit")):
        logger.info("rdkit is already installed")
        if not force:
            return

        logger.info("force re-install")

    url = url_base + file_name
    python_version = "{0}.{1}.{2}".format(*sys.version_info)

    logger.info("python version: {}".format(python_version))

    if os.path.isdir(conda_path):
        logger.warning("remove current miniconda")
        shutil.rmtree(conda_path)
    elif os.path.isfile(conda_path):
        logger.warning("remove {}".format(conda_path))
        os.remove(conda_path)

    logger.info('fetching installer from {}'.format(url))
    res = requests.get(url, stream=True)
    res.raise_for_status()
    with open(file_name, 'wb') as f:
        for chunk in res.iter_content(chunk_size):
            f.write(chunk)
    logger.info('done')

    logger.info('installing miniconda to {}'.format(conda_path))
    subprocess.check_call(["bash", file_name, "-b", "-p", conda_path])
    logger.info('done')

    logger.info("installing rdkit")
    subprocess.check_call([
        os.path.join(conda_path, "bin", "conda"),
        "install",
        "--yes",
        "-c", "rdkit",
        "python==3.7.3",
        "rdkit" if rdkit_version is None else "rdkit=={}".format(rdkit_version)])
    logger.info("done")

    import rdkit
    logger.info("rdkit-{} installation finished!".format(rdkit.__version__))


if __name__ == "__main__":
    install()

In [ ]:
# If something breaks in the notebook it is probably related to a mismatch between the Python version, CUDA or torch
import torch
pytorch_version = f"torch-{torch.__version__}.html"
!pip install --no-index torch-scatter -f https://pytorch-geometric.com/whl/$pytorch_version
!pip install --no-index torch-sparse -f https://pytorch-geometric.com/whl/$pytorch_version
!pip install --no-index torch-cluster -f https://pytorch-geometric.com/whl/$pytorch_version
!pip install --no-index torch-spline-conv -f https://pytorch-geometric.com/whl/$pytorch_version
!pip install torch-geometric

In [ ]:
import torch
import sys

# Aim for a specific stable PyTorch version that is well-supported by PyTorch Geometric
TARGET_TORCH_VERSION = "2.2.2"

print(f"Current PyTorch version detected: {torch.__version__}")

# Check if the current PyTorch version is the target version (ignoring +cpu or +cuXXX suffix)
current_base_version = torch.__version__.split('+')[0]
if current_base_version != TARGET_TORCH_VERSION:
    print(f"PyTorch version is {current_base_version}, but target is {TARGET_TORCH_VERSION}. Reinstalling PyTorch...")

    # Uninstall existing torch and torchvision
    !pip uninstall torch -y -q
    !pip uninstall torchvision -y -q

    # Determine if a GPU is available to install the correct PyTorch wheels
    if torch.cuda.is_available():
        cuda_version_short = torch.version.cuda.replace('.', '') # e.g., '118' or '121'
        # PyTorch wheels are named like torch-2.2.2+cu118
        cuda_package_suffix = f"cu{cuda_version_short}"
        index_url = f"https://download.pytorch.org/whl/{cuda_package_suffix}"
        print(f"Installing torch=={TARGET_TORCH_VERSION}+{cuda_package_suffix} and compatible torchvision for CUDA from {index_url}...")
        !pip install torch=={TARGET_TORCH_VERSION}+{cuda_package_suffix} torchvision==0.17.2+{cuda_package_suffix} --index-url {index_url} -q
    else:
        print(f"CUDA not detected. Installing torch=={TARGET_TORCH_VERSION}+cpu and compatible torchvision for CPU...")
        !pip install torch=={TARGET_TORCH_VERSION}+cpu torchvision==0.17.2+cpu --index-url https://download.pytorch.org/whl/cpu -q

    # Re-import torch after installation to reflect the new version
    import torch
    print(f"PyTorch version after re-installation: {torch.__version__}")
else:
    print(f"PyTorch version {TARGET_TORCH_VERSION} is already installed.")

# Check environment details (main kernel's `torch` import is most reliable)
print(f"PyTorch version in main kernel: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA available. Version: {torch.version.cuda}")
else:
    print("CUDA not available. Using CPU.")
print(f"Python version: {sys.version.split(' ')[0]}")

In [ ]:
# Install rdkit using the correct pip package name
!pip install rdkit -q
import rdkit
print(f"RDKit version: {rdkit.__version__}")

In [ ]:
import torch

# Get the base PyTorch version (e.g., '2.1.0' from '2.1.0+cpu')
TORCH = torch.__version__.split('+')[0]

# Determine CUDA version for PyTorch Geometric wheels, or default to 'cpu'
CUDA = 'cpu'
if torch.cuda.is_available():
    # torch.version.cuda might be '11.8' or '12.1'
    # PyG expects 'cu118' or 'cu121'
    cuda_version_str = torch.version.cuda.replace('.', '')
    if cuda_version_str.isdigit():
        CUDA = 'cu' + cuda_version_str
    else:
        print(f"Warning: Unexpected CUDA version format: {torch.version.cuda}. Defaulting to CPU wheels for PyG.")

print(f"Detected PyTorch base version for PyG wheels: {TORCH}, Detected CUDA version for PyG wheels: {CUDA}")

# Install torch-geometric and its dependencies using the determined versions
# Use -q for quiet install and -f to specify the index URL
!pip install --no-index torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-{TORCH}/{CUDA}.html -q

In [ ]:
import rdkit
from torch_geometric.datasets import MoleculeNet

# Load the ESOL dataset
data = MoleculeNet(root=".", name="ESOL")
data

In [ ]:
# Investigating the dataset
print("Dataset type: ", type(data))
print("Dataset features: ", data.num_features)
print("Dataset target: ", data.num_classes)
print("Dataset length: ", data.len)
print("Dataset sample: ", data[0])
print("Sample  nodes: ", data[0].num_nodes)
print("Sample  edges: ", data[0].num_edges)

# edge_index = graph connections
# smiles = molecule with its atoms
# x = node features (32 nodes have each 9 features)
# y = labels (dimension)

In [ ]:
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
embedding_size = 64

class GCN(torch.nn.Module):
    def __init__(self):
        # Init parent
        super(GCN, self).__init__()
        torch.manual_seed(42)

        # GCN layers
        self.initial_conv = GCNConv(data.num_features, embedding_size)
        self.conv1 = GCNConv(embedding_size, embedding_size)
        self.conv2 = GCNConv(embedding_size, embedding_size)
        self.conv3 = GCNConv(embedding_size, embedding_size)

        # Output layer
        self.out = Linear(embedding_size*2, 1)

    def forward(self, x, edge_index, batch_index):
        # First Conv layer
        hidden = self.initial_conv(x, edge_index)
        hidden = F.tanh(hidden)

        # Other Conv layers
        hidden = self.conv1(hidden, edge_index)
        hidden = F.tanh(hidden)
        hidden = self.conv2(hidden, edge_index)
        hidden = F.tanh(hidden)
        hidden = self.conv3(hidden, edge_index)
        hidden = F.tanh(hidden)

        # Global Pooling (stack different aggregations)
        hidden = torch.cat([gmp(hidden, batch_index),
                            gap(hidden, batch_index)], dim=1)

        # Apply a final (linear) classifier.
        out = self.out(hidden)

        return out, hidden

model = GCN()
print(model)
print("Number of parameters: ", sum(p.numel() for p in model.parameters()))

In [ ]:
from torch_geometric.data import DataLoader
import warnings
warnings.filterwarnings("ignore")

# Root mean squared error
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0007)

# Use GPU for training
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Wrap data in a data loader
data_size = len(data)
NUM_GRAPHS_PER_BATCH = 64
loader = DataLoader(data[:int(data_size * 0.8)],
                    batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
test_loader = DataLoader(data[int(data_size * 0.8):],
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)

def train(data):
    # Enumerate over the data
    for batch in loader:
      # Use GPU
      batch.to(device)
      # Reset gradients
      optimizer.zero_grad()
      # Passing the node features and the connection info
      pred, embedding = model(batch.x.float(), batch.edge_index, batch.batch)
      # Calculating the loss and gradients
      loss = loss_fn(pred, batch.y)
      loss.backward()
      # Update using the gradients
      optimizer.step()
    return loss, embedding

print("Starting training...")
losses = []
for epoch in range(2000):
    loss, h = train(data)
    losses.append(loss)
    if epoch % 100 == 0:
      print(f"Epoch {epoch} | Train Loss {loss}")

In [ ]:
# Investigating the features
# Shape: [num_nodes, num_node_features]
data[0].x

In [ ]:
# Investigating the edges in sparse COO format
# Shape [2, num_edges]
data[0].edge_index.t()

In [ ]:
data[0]["smiles"]

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
molecule = Chem.MolFromSmiles(data[0]["smiles"])
molecule

In [ ]:
type(molecule)

In [ ]:
# Visualize learning (training loss)
import seaborn as sns
losses_float = [float(loss.cpu().detach().numpy()) for loss in losses]
loss_indices = [i for i,l in enumerate(losses_float)]
plt = sns.lineplot(loss_indices, losses_float)
plt

In [ ]:
# Visualize learning (training loss)
import seaborn as sns
losses_float = [float(loss.cpu().detach().numpy()) for loss in losses]
loss_indices = [i for i,l in enumerate(losses_float)]
plt = sns.lineplot(loss_indices, losses_float)
plt

In [ ]:
plt = sns.scatterplot(data=df, x="y_real", y="y_pred")
plt.set(xlim=(-7, 2))
plt.set(ylim=(-7, 2))
plt

In [ ]:
plt = sns.scatterplot(data=df, x="y_real", y="y_pred")
plt.set(xlim=(-7, 2))
plt.set(ylim=(-7, 2))
plt